## Cleaning Module — Usage Example

This example demonstrates how to use the cleaning utilities from the pyspark_utils library to prepare raw insurance policy data for downstream processing.

It covers common real‑world tasks such as:

- Standardizing column names
- Cleaning text fields
- Handling null values
- Normalizing date formats
- Casting numeric fields
- Deduplicating records

### Import Libraries
-----

In [ ]:
from pyspark.sql import SparkSession
from pyspark_utils.cleaning import (
    column_names,
    nulls,
    deduplication,
    dates,
    type_casting,
    text_cleaning
)

### Cleaning Raw Insurance Policy Data
---

In [ ]:
spark = (
    SparkSession.builder
    .appName("PolicyCleaningExample")
    .getOrCreate()
)

raw_data = [
    ("  POL123 ", "John   Doe", "2024/01/10", None, "1000", "ACTIVE"),
    ("POL123", "John Doe", "2024-01-10", None, "1000", "ACTIVE"),
    ("POL456", "Mary   Smith", "10-02-2024", "2024-12-31", "2000", "ACTIVE"),
    ("POL789", None, "2024-03-01", "2024-12-31", None, "PENDING")
]

columns = ["policy_number", "customer_name", "start_date", "end_date", "premium", "status"]

df = spark.createDataFrame(raw_data, columns)


### Step-by-Step Cleaning
---

In [ ]:
# Standardize Column Names

df = column_names.standardize_column_names(df, case="snake")


# Clean Text Fields

df = text_cleaning.trim_strings(df)
df = text_cleaning.normalize_whitespace(df, ["customer_name"])
df = text_cleaning.normalize_whitespace(df, ["customer_name", "status"])


# Remove special characters from multiple columns
df = text_cleaning.remove_special_characters(
    df,
    ["customer_name", "policy_number"]
)


# Standardize Date Formats
df = dates.standardize_date_formats(
    df,
    columns=["start_date", "end_date"],
    fmt="yyyy-MM-dd"
)


# Handle Null Values
df = nulls.fill_nulls(df, {
    "premium": "0",
    "customer_name": "UNKNOWN"
})


df = nulls.fill_nulls(
    df,
    {
        "customer_name": "UNKNOWN",
        "premium": "0",
        "status": "PENDING"
    }
)



# Cast Numeric Fields

df = type_casting.cast_columns(df, {"premium": "double"})

df = type_casting.cast_columns(
    df,
    {
        "premium": "double",
        "coverage_amount": "double"
    }
)


# Deduplicate Policies
df = deduplication.dedupe(
    df,
    keys=["policy_number"],
    order_by="start_date"
)


### ✅ Final Output

After applying all cleaning steps, the DataFrame is ready for:

- Delta Lake ingestion
- Data quality checks
- Feature engineering
- Downstream transformations